In [ ]:
import pandas as pd
from json import loads


temperatura_novo = []
load_novo = []
otimizacao_novo = []
size_novo = []

OTIMIZACOES=("O0", "O3", "O3-native")
SIZES=(1000, 3000, 5000)

for otimizacao in OTIMIZACOES:
    for size in SIZES:
        for rep in range(1,4):
            df=pd.read_csv(f"coleta_novo/{otimizacao}_{size}_rep{rep}_monitoramento.csv", sep=",")
            load_novo.append(float(df["loadavg"].at[0].split("'")[1]))
            temperatura_novo.append(loads(df["temperatura_C"].at[0])["k10temp_temp1_input"])
            otimizacao_novo.append(otimizacao)
            size_novo.append(size)


df_condicoes_novo = pd.DataFrame({"temperatura": temperatura_novo, "carga": load_novo, "otimizacao":otimizacao_novo, "tamanho_matriz":size_novo})
df_condicoes_novo["computador"] = "novo"
df_condicoes_novo.groupby(["computador", "otimizacao", "tamanho_matriz"]).agg(["mean", "std"])

temperatura               carga          
                                            mean       std      mean       std
computador otimizacao tamanho_matriz                                          
novo       O0         1000             40.400000  0.400000  0.790000  0.185203
                      3000             45.433333  3.932345  0.753333  0.035119
                      5000             52.033333  1.817507  0.913333  0.055076
           O3         1000             50.866667  3.008876  1.973333  0.317857
                      3000             45.166667  0.665833  1.913333  0.122202
                      5000             45.066667  0.808290  1.720000  0.098489
           O3-native  1000             44.766667  1.357694  1.436667  0.120139
                      3000             42.800000  0.360555  1.116667  0.090185
                      5000             44.500000  0.888819  0.870000  0.070000

In [ ]:
temperatura_antigo1 = []
temperatura_antigo2 = []

load_antigo = []
otimizacao_antigo = []
size_antigo = []

OTIMIZACOES = ("O0", "O3", "O3-native")
SIZES = (1000, 3000, 5000)

for otimizacao in OTIMIZACOES:
    for size in SIZES:
        for rep in range(1, 4):
            df = pd.read_csv(f"coleta_antigo/{otimizacao}_{size}_rep{rep}_monitoramento.csv", sep=",")
            load_antigo.append(float(df["loadavg"].at[0].split("'")[1]))
            temperatura_antigo1.append(loads(df["temperatura_C"].at[0])["coretemp_temp1_input"])
            temperatura_antigo2.append(loads(df["temperatura_C"].at[0])["coretemp_temp2_input"])

            otimizacao_antigo.append(otimizacao)
            size_antigo.append(size)


df_condicoes_antigo = pd.DataFrame({"temperatura_c1": temperatura_antigo1, "temperatura_c2": temperatura_antigo2, "carga": load_antigo, "otimizacao": otimizacao_antigo, "tamanho_matriz": size_antigo})
df_condicoes_antigo["computador"] = "antigo"
df_condicoes_antigo.groupby(["computador", "otimizacao", "tamanho_matriz"]).agg(["mean", "std"])


temperatura_c1           temperatura_c2  \
                                               mean       std           mean   
computador otimizacao tamanho_matriz                                           
antigo     O0         1000                38.333333  0.577350      38.333333   
                      3000                41.333333  2.516611      40.666667   
                      5000                45.666667  4.041452      45.666667   
           O3         1000                47.000000  3.605551      45.666667   
                      3000                47.333333  3.214550      46.333333   
                      5000                44.000000  0.000000      43.000000   
           O3-native  1000                47.333333  2.516611      47.333333   
                      3000                45.000000  1.000000      43.666667   
                      5000                45.000000  1.732051      44.000000   

                                                   carga            
                                           std      mean       std  
computador otimizacao tamanho_matriz                                
antigo     O0         1000            0.577350  1.896667  0.160104  
                      3000            2.081666  1.506667  0.090185  
                      5000            4.041452  2.293333  0.497024  
           O3         1000            3.055050  2.170000  0.141067  
                      3000            2.516611  2.316667  0.283784  
                      5000            0.000000  2.200000  0.108167  
           O3-native  1000            2.516611  2.073333  0.175024  
                      3000            1.154701  1.820000  0.026458  
                      5000            1.732051  1.836667  0.367469

In [4]:
from subprocess import run
from pathlib import Path
from sys import platform
from string import Template


SIZES: tuple[int] = (1000, 2000, 3000, 4000, 5000)
FOLDERS_OTIMIZATION_LEVEL: tuple[str] = (
    ("otimizacao-00", "00"),
    ("otimizacao-03", "03"),
    ("otimizacao-03-native", "03-native"),
)
result = []
otm_level = []
sizes = []
pc_used = []
PC = (("antigo","pc_antigo"), ("novo", "pc_novo"),)

file_name: Template = Template(
    "${experimented_pc}/${folder}/solution_${otimization}_${size}_${run}.bin"
    if platform == "linux"
    else "${experimented_pc}\\${folder}\\solution_${otimization}_${size}_${run}.bin"
)
for pc, pc_folder in PC:
    for folder, otimization_lv in FOLDERS_OTIMIZATION_LEVEL:
        for size in SIZES:
            for idx in range(1, 11):
                file: Path = Path(
                    file_name.substitute(
                        experimented_pc=pc_folder, folder=folder, otimization=otimization_lv, size=size, run=idx
                    )
                )
                pc_used.append(pc)
                sizes.append(size)
                otm_level.append(otimization_lv)
                is_valid = run(["../fontes/validacao/diff-mat.exec", "1E-10", f"../fontes/X{size}.bin", file], capture_output=True, text=True)
                result.append(True if "true" in is_valid.stdout else False)

df_validation = pd.DataFrame({"computer_used": pc_used, "matrix_size":sizes, "otization_level":otm_level, "valid_result":result})
print("Todas matrizes estão válidas\n" if df_validation["valid_result"].all() else "Nenhuma matriz está valida\n")
print("\n==========================  Exemplificação Antigo  ==============================")
print(df_validation.head(50))
print("\n==========================   Exemplificação Novo   ==============================\n")
print(df_validation.tail(50))

Todas matrizes estão válidas


==========================  Exemplificação Antigo  ==============================
   computer_used  matrix_size otization_level  valid_result
0         antigo         1000              00          True
1         antigo         1000              00          True
2         antigo         1000              00          True
3         antigo         1000              00          True
4         antigo         1000              00          True
5         antigo         1000              00          True
6         antigo         1000              00          True
7         antigo         1000              00          True
8         antigo         1000              00          True
9         antigo         1000              00          True
10        antigo         2000              00          True
11        antigo         2000              00          True
12        antigo         2000              00          True
13        antigo         2000              00  